In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, io, json, shutil, importlib
sys.path.append("/content/drive/MyDrive/tcc")
from src import config, ela
importlib.reload(ela)

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

v2 = pd.read_csv(f"{config.DRIVE}/dataset_v2.csv")
FATOR = json.load(open(f"{config.ELA_DIR}/etapa1/fator_ela.json"))["fator"]
T, N = config.CROP, config.CROPS_TREINO
print("Fator do ELA:", FATOR)
print(v2.groupby(["grupo", "classe"]).size())

Mounted at /content/drive
Fator do ELA: 36.43
grupo        classe
base         ia        1000
             real      1000
dispositivo  real       208
firefly      ia        1000
ia_extra     ia         208
dtype: int64


In [2]:
def recomprimir(img, q):
    """Recomprime a imagem em memória como JPEG com qualidade q."""
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=int(q), subsampling=2)
    buf.seek(0)
    with Image.open(buf) as out:
        return out.convert("RGB")


def processar(sub, nome, comprimir, regiao_central, modo_crop, seed):
    """Gera os crops de ELA e RGB de um conjunto e salva em DEST."""
    if AMOSTRA:
        sub = sub.groupby("classe").head(AMOSTRA)
    arq_ela = f"{DEST}/{nome}_ela.npy"
    if os.path.exists(arq_ela):
        print(f"{nome}: já existe, pulando")
        return

    rng = np.random.default_rng(seed)
    total = len(sub) * N
    tmp_ela = np.lib.format.open_memmap(f"/content/{nome}_ela.npy", mode="w+", dtype=np.uint8, shape=(total, T, T, 3))
    tmp_rgb = np.lib.format.open_memmap(f"/content/{nome}_rgb.npy", mode="w+", dtype=np.uint8, shape=(total, T, T, 3))

    rotulos, k = [], 0
    for _, r in tqdm(sub.iterrows(), total=len(sub), desc=nome):
        with Image.open(f"{config.DRIVE}/{r['arquivo']}") as im:
            img = im.convert("RGB")

        q = None
        if comprimir:
            q = int(rng.integers(70, 99))
            img = recomprimir(img, q)

        rgb = np.asarray(img)
        e = ela.escalar(ela.calcular_ela(img, config.ELA_QUALIDADE), FATOR)

        h, w = rgb.shape[:2]
        if regiao_central and min(h, w) > 1024:
            oy, ox, rh, rw = (h - 1024) // 2, (w - 1024) // 2, 1024, 1024
        else:
            oy, ox, rh, rw = 0, 0, h, w

        if modo_crop(r) == "aleatorio":
            posicoes = ela.posicoes_aleatorias(rh, rw, T, N, rng)
        else:
            posicoes = ela.posicoes_grade(rh, rw, T, N)

        for y, x in posicoes:
            y, x = y + oy, x + ox
            tmp_ela[k] = e[y:y + T, x:x + T]
            tmp_rgb[k] = rgb[y:y + T, x:x + T]
            rotulos.append({
                "indice": k, "id": r["id"], "classe": r["classe"],
                "rotulo": int(r["classe"] == "ia"),
                "aparelho_ou_gerador": r["aparelho_ou_gerador"], "grupo": r["grupo"],
                "particao_foto": r.get("particao_foto"), "qualidade_jpeg": q, "y": y, "x": x,
            })
            k += 1

    tmp_ela.flush(); tmp_rgb.flush()
    del tmp_ela, tmp_rgb
    shutil.copy(f"/content/{nome}_rgb.npy", f"{DEST}/{nome}_rgb.npy")
    shutil.copy(f"/content/{nome}_ela.npy", arq_ela)
    os.remove(f"/content/{nome}_ela.npy"); os.remove(f"/content/{nome}_rgb.npy")
    pd.DataFrame(rotulos).to_csv(f"{DEST}/{nome}_rotulos.csv", index=False)
    print(f"{nome}: {k} crops salvos")

In [5]:
AMOSTRA = None      # troque para None para rodar com tudo

DEST = f"{config.ELA_DIR}/{'v2_amostra' if AMOSTRA else 'v2'}"
os.makedirs(DEST, exist_ok=True)

base = v2[v2["grupo"] == "base"]
disp = v2[v2["grupo"] == "dispositivo"]

processar(base[base["split"] == "treino"], "treino_base", True, False, lambda r: "aleatorio", config.SEED)
processar(base[base["split"] == "val"],    "val_base",    True, False, lambda r: "grade",     config.SEED + 1)
processar(v2[v2["grupo"] == "ia_extra"],   "ia_extra",    True, False, lambda r: "aleatorio", config.SEED + 2)

for i, aparelho in enumerate(sorted(disp["aparelho_ou_gerador"].unique())):
    d = disp[disp["aparelho_ou_gerador"] == aparelho]
    processar(d, f"disp_aug_{aparelho}", True, True,
              lambda r: "aleatorio" if r["particao_foto"] == "treino" else "grade", config.SEED + 10 + i)
    processar(d, f"disp_nativo_{aparelho}", False, True, lambda r: "grade", config.SEED + 20 + i)

treino_base: 100%|██████████| 1400/1400 [22:55<00:00,  1.02it/s]


treino_base: 5600 crops salvos


val_base: 100%|██████████| 300/300 [04:49<00:00,  1.04it/s]


val_base: 1200 crops salvos


ia_extra: 100%|██████████| 208/208 [03:15<00:00,  1.06it/s]


ia_extra: 832 crops salvos


... (8 outputs left out)

disp_aug_samsung_s23fe: 100%|██████████| 44/44 [01:39<00:00,  2.27s/it]


disp_aug_samsung_s23fe: 176 crops salvos


disp_nativo_samsung_s23fe: 100%|██████████| 44/44 [00:47<00:00,  1.08s/it]


disp_nativo_samsung_s23fe: 176 crops salvos


In [6]:
for nome in sorted(f for f in os.listdir(DEST) if f.endswith("_ela.npy")):
    arr = np.load(f"{DEST}/{nome}", mmap_mode="r")
    rot = pd.read_csv(f"{DEST}/{nome.replace('_ela.npy', '_rotulos.csv')}")
    q = rot["qualidade_jpeg"].dropna()
    faixa_q = f"q {int(q.min())}–{int(q.max())}" if len(q) else "nativo"
    print(f"{nome:38s} {str(arr.shape):22s} reais={(rot.rotulo == 0).sum():5d}  ia={(rot.rotulo == 1).sum():5d}  {faixa_q}")

disp_aug_canon_powershot_sx510hs_ela.npy (260, 256, 256, 3)     reais=  260  ia=    0  q 70–98
disp_aug_samsung_a56_ela.npy           (396, 256, 256, 3)     reais=  396  ia=    0  q 70–98
disp_aug_samsung_s23fe_ela.npy         (176, 256, 256, 3)     reais=  176  ia=    0  q 70–98
disp_nativo_canon_powershot_sx510hs_ela.npy (260, 256, 256, 3)     reais=  260  ia=    0  nativo
disp_nativo_samsung_a56_ela.npy        (396, 256, 256, 3)     reais=  396  ia=    0  nativo
disp_nativo_samsung_s23fe_ela.npy      (176, 256, 256, 3)     reais=  176  ia=    0  nativo
ia_extra_ela.npy                       (832, 256, 256, 3)     reais=    0  ia=  832  q 70–98
treino_base_ela.npy                    (5600, 256, 256, 3)    reais= 2800  ia= 2800  q 70–98
val_base_ela.npy                       (1200, 256, 256, 3)    reais=  600  ia=  600  q 70–98
